In [10]:
import aiohttp
import asyncio
import nest_asyncio
import os
from dotenv import load_dotenv
from typing import List, Dict, Any, Tuple, Union
import json
from github import Github, Auth
from gidgethub.aiohttp import GitHubAPI
from gidgethub import GitHubException
import sys
import base64
from stamina import retry, retry_context
import re
from functools import partial
from textacy import preprocessing
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from qdrant_client.http.models import Distance, VectorParams
import uuid
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langgraph.graph import START, StateGraph
from langchain_core.documents import Document
from typing_extensions import List, TypedDict

import numpy as np
import matplotlib.pyplot as plt

# allow for multiple event loops within Jupyter
nest_asyncio.apply()

# load environment variables
load_dotenv()

import pickle

In [14]:
def _is_retriable(exc: Exception) -> bool:
    """Retry on transient network/server errors; skip deterministic 404/403."""
    if isinstance(exc, GitHubException):
        return exc.status_code not in (404, 403)
    return isinstance(exc, aiohttp.ClientError)


@retry(on=_is_retriable, attempts=3, wait_initial=0.5, wait_max=10.0)
async def get_root_markdown_files(gh: GitHubAPI, owner: str, repo: str) -> List[Dict[Any, Any]]:
    """
    Get all markdown files from the root directory of a repository.
    This includes README.md, CONTRIBUTING.md, CHANGELOG.md, etc.
    """
    try:
        # Get contents of root directory
        contents = await gh.getitem(f"/repos/{owner}/{repo}/contents/")

        # Filter for markdown files (case-insensitive)
        markdown_files = [
            file for file in contents
            if file['type'] == 'file' and file['name'].lower().endswith('.md')
        ]

        return markdown_files
    except GitHubException as e:
        print(f"Error fetching contents for {owner}/{repo}: {e}")
        return []


@retry(on=_is_retriable, attempts=3, wait_initial=0.5, wait_max=10.0)
async def fetch_markdown_content(gh: GitHubAPI, owner: str, repo: str, file_path: str):
    """Fetch content of a specific markdown file"""
    try:
        file_data = await gh.getitem(f"/repos/{owner}/{repo}/contents/{file_path}")
        content = base64.b64decode(file_data['content']).decode('utf-8')

        return {
            'name': file_data['name'],
            'path': file_data['path'],
            'size': file_data['size'],
            'content': content,
            'success': True
        }
    except GitHubException as e:
        return {
            'path': file_path,
            'success': False,
            'error': str(e)
        }


async def fetch_starred_repos_with_docs(max_repos: int = None, concurrent_tasks: int = 20) -> List[Dict[Any, Any]]:
    """
    Fetch all starred repositories for the authenticated GitHub user, then
    concurrently retrieve documentation for each repo using the following strategy:

      1. Try the /readme endpoint first (covers README.md, README.rst, etc.)
      2. If no README exists (404), fall back to fetching ALL markdown files
         found in the root directory of the repository.
      3. If neither exists, the repo is recorded with an empty docs list.

    All per-repo fetches run concurrently via asyncio.gather, so even a large
    starred list is handled as fast as the GitHub rate limit allows.

    Args:
        max_repos: Optional cap on the number of starred repos to process.
                   Defaults to None (fetch all starred repos).

    Returns:
        A list of dicts, one per repo, with keys:
            repo         - "owner/name"
            description  - repo description string
            stars        - stargazer count
            language     - primary language
            url          - HTML URL
            doc_source   - "readme" | "root_markdown" | None
            docs         - list of {name, path, size, content} dicts
    """
    oauth_token = os.getenv("GITHUB_TOKEN")

    # Semaphore pattern for throttling concurrent asyncio tasks
    semaphore = asyncio.Semaphore(concurrent_tasks)

    async with aiohttp.ClientSession() as session:
        gh = GitHubAPI(session, "markdown-fetcher", oauth_token=oauth_token)

        # ── Step 1: Collect starred repos (sequential; getiter handles pagination) ──
        print("Fetching starred repositories...")
        starred_repos: List[Dict[Any, Any]] = []
        async for repo in gh.getiter("/user/starred", accept="application/vnd.github.mercy-preview+json"):
            starred_repos.append(repo)
            if max_repos and len(starred_repos) >= max_repos:
                break

        print(f"Found {len(starred_repos)} starred repositories")
        print("Fetching documentation for all repos concurrently...\n")

        # ── Step 2: Define per-repo coroutine ──────────────────────────────────────
        async def fetch_repo_docs(repo: Dict[Any, Any]) -> Dict[Any, Any]:
            owner = repo["owner"]["login"]
            name = repo["name"]
            full_name = repo["full_name"]
            topics: List[str] = repo["topics"]

            base = {
                "repo": full_name,
                "description": repo.get("description"),
                "topics": topics,
                "stars": repo.get("stargazers_count"),
                "language": repo.get("language"),
                "url": repo.get("html_url"),
            }

            # Try README first via the dedicated /readme endpoint
            try:
                async for attempt in retry_context(
                    on=_is_retriable, attempts=3, wait_initial=0.5, wait_max=10.0
                ):
                    with attempt:
                        readme_data = await gh.getitem(f"/repos/{owner}/{name}/readme")
                content = base64.b64decode(readme_data["content"]).decode("utf-8")
                return {
                    **base,
                    "doc_source": "readme",
                    "docs": [
                        {
                            "name": readme_data["name"],
                            "path": readme_data["path"],
                            "size": readme_data["size"],
                            "content": content,
                        }
                    ],
                }
            except GitHubException as e:
                if e.status_code != 404:
                    # Surface unexpected errors without crashing the whole gather
                    print(f"  Warning: unexpected error fetching README for {full_name}: {e}")

            # README absent — fall back to all root-level .md files
            markdown_files = await get_root_markdown_files(gh, owner, name)
            if markdown_files:
                tasks = [
                    fetch_markdown_content(gh, owner, name, f["path"])
                    for f in markdown_files
                ]
                file_results = await asyncio.gather(*tasks)
                return {
                    **base,
                    "doc_source": "root_markdown",
                    "docs": [r for r in file_results if r.get("success")],
                }

            # No documentation found at all
            return {**base, "doc_source": None, "docs": []}

        # ── Step 3: Fan out — all repos fetched concurrently ──────────────────────
        async def fetch_repo_docs_throttled(repo: Dict[Any, Any]) -> Dict[Any, Any]:
            async with semaphore:
                return await fetch_repo_docs(repo)

        results: List[Dict[Any, Any]] = await asyncio.gather(
            *[fetch_repo_docs_throttled(repo) for repo in starred_repos]
        )

        # ── Summary ────────────────────────────────────────────────────────────────
        with_readme = [r for r in results if r["doc_source"] == "readme"]
        with_md = [r for r in results if r["doc_source"] == "root_markdown"]
        no_docs = [r for r in results if r["doc_source"] is None]

        print(f"\n{'='*70}")
        print("DOCUMENTATION FETCH SUMMARY")
        print(f"{'='*70}")
        print(f"Total repos processed : {len(results)}")
        print(f"  README found        : {len(with_readme)}")
        print(f"  Root markdown files : {len(with_md)}")
        print(f"  No docs found       : {len(no_docs)}")
        if gh.rate_limit:
            print(f"Rate limit remaining  : {gh.rate_limit.remaining}")

        return results


def strip_markdown(text: str) -> str:
    """Remove common Markdown syntax; keep inner text and emojis."""
    # Headings: ## Title -> Title
    text = re.sub(r"^#{1,6}\s*", "", text, flags=re.MULTILINE)
    # Inline code: `code` -> code
    text = re.sub(r"`([^`]+)`", r"\1", text)
    # Bold/italic: **bold** or *italic* -> bold / italic
    text = re.sub(r"\*{1,2}([^*]+)\*{1,2}", r"\1", text)
    text = re.sub(r"_{1,2}([^_]+)_{1,2}", r"\1", text)
    # Link markup: [text](url) -> text
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)
    # Collapse many newlines
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text


def make_normalize_text_pipeline(
    *,
    url_repl: str = "",
    unicode_form: str = "NFC",
):
    """
    Build a textacy pipeline: Markdown/HTML → plain text (emojis kept).

    Args:
        url_repl: Replacement for URLs (default ""). Use "_URL_" to keep a placeholder.
        unicode_form: Unicode normalization form ("NFC", "NFD", "NFKC", "NFKD"). Default "NFC".

    Returns:
        A callable that takes a string and returns preprocessed plain text.
    """
    return preprocessing.make_pipeline(
        strip_markdown,
        preprocessing.remove.html_tags,
        preprocessing.normalize.bullet_points,
        preprocessing.normalize.quotation_marks,
        partial(preprocessing.normalize.unicode, form=unicode_form),
        preprocessing.normalize.whitespace,
    )


# Default pipeline instance
normalize_text_pipeline = make_normalize_text_pipeline()

In [6]:
MAX_REPOS = 3000

starred_repo_data = asyncio.run(fetch_starred_repos_with_docs(MAX_REPOS))

Fetching starred repositories...
Found 2049 starred repositories
Fetching documentation for all repos concurrently...


DOCUMENTATION FETCH SUMMARY
Total repos processed : 2049
  README found        : 2041
  Root markdown files : 0
  No docs found       : 8
Rate limit remaining  : 2874


In [7]:
# save the loaded github data from my account to pickled file
import pickle

with open("github_data.pkl", "wb") as f:
    pickle.dump(starred_repo_data, f)

In [8]:
# define embedding model for vector store
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
MAX_CHARACTERS = 30_000  # fall-back for when text is too long

In [24]:
NAMESPACE = uuid.NAMESPACE_URL

def repo_to_uuid(repo_name: str) -> str:
    return str(uuid.uuid5(NAMESPACE, repo_name))

def normalize_docs(docs: List[Dict[str, Any]]):
    content_str = ''
    for doc in docs:
        content_str += (doc['content'] + '\n\n')
    truncated = content_str[:MAX_CHARACTERS] if len(content_str) > MAX_CHARACTERS else content_str
    normalized_text = normalize_text_pipeline(truncated)
    return normalized_text


def organize_repo_data(repo_data: List[Dict[Any, Any]]) -> Tuple[List[str], List[Dict[str, Union[str, int]]], List[str]]:
    ids = []
    metadata = []
    docs = []
    for repo in repo_data:
        id = repo_to_uuid(repo['repo'])
        description = repo['description']
        topics = repo['topics']
        doc_source = repo['doc_source']
        stars = repo['stars']
        url = repo['url']
        language = repo['language']
        # append to lists for DB consumption
        ids.append(id)
        metadata.append({
            'id': id,
            'repo': repo['repo'],
            'description': description,
            'topics': topics,
            'language': language,
            'doc_source': doc_source,
            'stars': stars,
            'url': url,
        })
        topics_str = f"Topics: {','.join(topics)}\n"
        docs.append(topics_str + normalize_docs(repo['docs']))
    return ids, metadata, docs

In [25]:
ids, metadata, docs = organize_repo_data(starred_repo_data)

In [26]:
print(f"id: {ids[0]}\n\nmetadata: {metadata[0]}\n\ndocs:\n {docs[0]}")

id: a5ef9890-4c52-5a83-8aa1-c1b266a4f6fb

metadata: {'id': 'a5ef9890-4c52-5a83-8aa1-c1b266a4f6fb', 'repo': 'pymc-devs/pytensor', 'description': 'PyTensor allows you to define, optimize, and efficiently evaluate mathematical expressions involving multi-dimensional arrays.', 'topics': ['ai', 'bayesian-inference', 'computational-science', 'deep-learning', 'statistics'], 'language': 'Python', 'doc_source': 'readme', 'stars': 594, 'url': 'https://github.com/pymc-devs/pytensor'}

docs:
 Topics: ai,bayesian-inference,computational-science,deep-learning,statistics
.. image:: https://cdn.rawgit.com/pymc-devs/pytensor/main/doc/images/PyTensorRGB.svg
 :height: 100px
 :alt: PyTensor logo
 :align: center
|Tests Status| |Coverage|
|Project Name| is a Python library that allows one to define, optimize, and
efficiently evaluate mathematical expressions involving multi-dimensional arrays.
It provides the computational backend for PyMC .
Features
- A hackable, pure-Python codebase
- Extensible graph fra

In [27]:
# build in-memory vector store # and load documents (repo content and metadata associated)
client = QdrantClient(":memory:")
client.create_collection(
    collection_name="ask_my_bookmark",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="ask_my_bookmark",
    embedding=embeddings,
)

_ = vector_store.add_texts(texts=docs, ids=ids, metadatas=metadata)
# define retriever
naive_retriever = vector_store.as_retriever(search_kwargs={"k" : 10})

In [28]:
### build RAG graph
def format_context(docs) -> str:
    chunks = []
    for doc in docs:
        meta = doc.metadata
        topics = meta.get('topics', [])
        if topics:
            topics_str = f"Topics: {','.join(topics)}"
        else:
            topics_str = "N/A"
        chunk = f"""---
Repo: {meta.get('repo', 'N/A')}
URL: {meta.get('url', 'N/A')}
Description: {meta.get('description', 'N/A')}
Topics: {topics_str}
Programming Language: {meta.get('language', 'N/A')}
Stars: {meta.get('stars', 'N/A')}

README excerpt:
{doc.page_content.strip()}
---"""
        chunks.append(chunk)
    return "\n\n".join(chunks)


# define the RAG Pipeline in LangChain
RAG_SYSTEM_PROMPT = """
You are AskMyBookmark, a personal research assistant with access to the user's GitHub starred repositories.

Your job is to help the user discover, recall, and explore repositories they have bookmarked on GitHub. You answer questions by reasoning over the retrieved repository context provided to you — not from your general knowledge of what exists on GitHub.

**Ground rules:**
- Only surface repositories that appear in the retrieved context below. Do not invent or suggest repositories that are not present in the context.
- If no retrieved repositories are relevant to the query, say so honestly and suggest the user try rephrasing or broadening their search.
- You may use your general knowledge to explain a topic or technology, but all repository recommendations must come exclusively from the retrieved context.

**When presenting results:**
- Always include the repository's full name (Repo) as a markdown link to its GitHub URL: [Repo](URL)
- Include a brief description of what the repo does (from the description and topics fields), written in your own words if the original description is terse or absent.
- Explain in 1–2 sentences *why* this repository is relevant to the user's query — this is the most important part.
- Group or rank results by relevance if there are several.
- If useful, note the primary programming language, star count, or topics to help the user evaluate the match.

**Tone:** Conversational, concise, and helpful. Treat the user as a developer who starred these repos intentionally and wants quick, intelligent recall — not a tutorial.

---

Retrieved repository context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template(RAG_SYSTEM_PROMPT),
    HumanMessagePromptTemplate.from_template("{question}"),
])
llm = ChatOpenAI(model="gpt-4.1-nano")

def generate(state):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = rag_prompt.format_messages(question=state["question"], context=docs_content)
    response = llm.invoke(messages)
    return {"response" : response.content}

class State(TypedDict):
    question: str
    context: List[Document]
    response: str

def retrieve(state):
    retrieved_docs = naive_retriever.invoke(state["question"])
    return {"context" : retrieved_docs}

# build graph
rag_graph_builder = StateGraph(State).add_sequence([retrieve, generate])
rag_graph_builder.add_edge(START, "retrieve")
rag_graph = rag_graph_builder.compile()

In [29]:
response = rag_graph.invoke({"question" : "Do I have any starred repositories related to Bayesian Statistics or Bayesian Modeling?"})
print(response['response'])

Yes, you have starred repositories related to Bayesian Statistics and Bayesian Modeling. Here are some of the relevant ones:

- [PyMC Resources](https://github.com/pyMC-devs/resources): This repository offers educational resources for PyMC, including ports of several Bayesian modeling books like "Bayesian Modeling and Computation in Python" and "Bayesian Statistical Methods." It is highly relevant if you are interested in Bayesian modeling in Python.
- [bayes-toolbox](https://github.com/kim-hyosub/bayes-toolbox): A Python package for running sophisticated Bayesian analyses in a straightforward way, developed by Hyosub E. Kim. It is tailored for Bayesian statistical workflows.
- [Bayesian Data Analysis course material](https://github.com/avehtari/BDAcourse): This repository contains course materials for Bayesian Data Analysis, based on the book by Gelman et al., with code examples in Python.
- [Bayesian Analysis with Python - Second Edition](https://github.com/astokk/Bayesian-Analysis-w

In [30]:
response2 = rag_graph.invoke({"question" : "What are some top deep learning libraries that I have starred on github?"})
print(response2['response'])

Based on your starred repositories, some top deep learning libraries you have bookmarked include:

- [Pixellib](https://github.com/ayoolaolaperson/PixelLib) — A library for performing segmentation of objects in images and videos, supporting both Mask R-CNN and PointRend architectures. It's relevant for deep learning applications in computer vision, especially instance and semantic segmentation.
- [Caffe](https://github.com/BVLC/caffe) — A deep learning framework designed for speed and modularity, particularly popular in computer vision tasks.
- [Deep Neural Networks library (Sonnet)](https://github.com/deepmind/sonnet) — TensorFlow-based neural network library by DeepMind for building complex models.
- [Flax](https://github.comGoogle/flax) — A neural network library built on JAX, optimized for flexibility and research.
- [TorchVision](https://github.com/pytorch/vision) — (implied in your list, often starred in conjunction with PyTorch) — a PyTorch package with models, datasets, and ima

In [31]:
response3 = rag_graph.invoke({"question" : "Can you tell me how many times the PyTorch Library has been starred on Github versus the Tensorflow library?"})
print(response3['response'])

Based on your starred repositories, the PyTorch library has been starred approximately 94K times on GitHub. The TensorFlow library, on the other hand, has about 200K stars. So, TensorFlow has significantly more stars than PyTorch in your starred repositories.


In [32]:
response4 = rag_graph.invoke({"question" : "What are the top ""awesome"" github repositories that I have starred?"})
print(response4['response'])

Based on the repositories you've starred, here are some of the top ones that stand out:

1. [Awesome-Kubernetes](https://github.com/sindresorhus/awesome)  
A curated list of Kubernetes resources, including guides, tools, and best practices. It's relevant if you're interested in container orchestration and cloud-native development.

2. [30 seconds of code](https://github.com/30secondsofcode/awesome)  
A collection of concise coding articles and snippets across multiple languages and topics, great for quick learning and reference.

3. [Flyte](https://github.com/flyteorg/flyte)  
An open-source cloud-native platform for building scalable data and machine learning pipelines on Kubernetes, perfect if you're into workflows and MLops.

4. [Awesome Hadoop !](https://github.com/sindresorhus/awesome)  
A comprehensive list of Hadoop and big data resources, useful if you're working with large-scale data processing.

5. [PyGitHub](https://github.com/PyGithub/PyGithub)  
A Python library to manage 

In [33]:
response5 = rag_graph.invoke({"question" : "What are the top machine learning repositories I have starred?"})
print(response5['response'])

Based on your starred repositories, here are some of the top machine learning-related repositories:

1. [Google Research](https://github.com/google-research/google-research) — Contains code from Google Research on various AI topics. It's a large collection of cutting-edge projects related to machine learning and deep learning, widely used for research and development.

2. [Complete Machine Learning Package](https://github.com/ageron/handson-ml) — A highly ranked, comprehensive repository with tutorials, notebooks, and tools for classical and deep machine learning, aimed at practical implementation.

3. [Libra](https://github.com/awslabs/libra) — A Python library that automates machine learning processes with one-liners, useful for quick model creation and evaluation.

4. [500+ Artificial Intelligence Project List](https://github.com/ashishpatel26/500-AI-Machine-learning-Deep-learning-Computer-vision-NLP-Projects-with-code) — A curated list of AI/ML projects across various domains inclu

In [34]:
response6 = rag_graph.invoke({"question" : "What are the top machine learning Python libraries I have starred? Please do not include repositories that are just educational resources or educational materials?"})
print(response6['response'])

Based on your starred repositories, the top machine learning Python libraries you have starred are:

- [NumPy](https://github.com/numpy/numpy) — The fundamental package for scientific computing with Python, widely used for array and matrix operations. Your star indicates you value its essential role in data manipulation and numerical computation in ML workflows.
- [scikit-learn](https://github.com/scikit-learn/scikit-learn) — A popular library for classical machine learning algorithms, providing simple and efficient tools for data mining and analysis. Your star reflects its importance in your ML projects.
- [TensorFlow](https://github.com/tensorflow/tensorflow) — An open-source deep learning framework used for building and deploying neural networks. Your starred TensorFlow repositories showcase your interest in deep learning and neural networks.
- [Keras](https://github.com/keras-team/keras) — A high-level neural networks API that runs on top of TensorFlow, making model design easier. 

In [35]:
response7 = rag_graph.invoke({"question" : "Do I have any R programming language libraries or packages starred on github?"})
print(response7['response'])

Yes, you have starred several R packages related to data analysis and manipulation. Notably:

- [tidypolars for R](https://github.com/robertjgraham/tidypolars): This package provides an interface to Polars, a fast DataFrame library, with a syntax familiar to R tidyverse users. It’s relevant if you're exploring fast DataFrame operations in R.

- [rpolars for R](https://github.com/robertjgraham/rpolars): This package allows using Polars DataFrames directly from R, offering a high-performance alternative to base R or data.table.

- [polarssql](https://github.com/robertjgraham/polarssql): An experimental package that provides a DBI interface to Polars, enabling SQL-based workflows in R with Polars.

- [r-polars-dashboard](https://github.com/robertjgraham/r-polars-dashboard): A dashboard tool comparing R and Python Polars APIs, useful for understanding cross-language features.

- [neo-r-polars](https://github.com/robertjgraham/neo-r-polars): The next-generation R API for Polars, focusing on

In [36]:
response8 = rag_graph.invoke({"question" : "Are there any R programming language libraries starred that are not related to data analysis?"})
print(response8['response'])

Based on the retrieved context, the R programming language repositories with stars are mostly related to data analysis and visualization. Specifically, the cheat sheets and data packages listed for R (like tidiverse, data.table, lubridate, and data import/transformation tools) focus on data analysis, cleaning, and visualization tasks.

There are no starred R repositories mentioned that are explicitly unrelated to data analysis, so it seems the starred R projects here are primarily centered on data analysis, visualization, and related tasks. Would you like to explore specific types of R libraries or datasets that might be outside data analysis?
